In [ ]:
import nltk
import numpy as np
from scipy.stats import randint, loguniform
from tensorflow.keras.preprocessing.text import Tokenizer 
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import Embedding, Bidirectional, Dense, Input, Dropout, Attention
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import tensorflow as tf 
from scikeras.wrappers import KerasClassifier
import pickle 
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd 

data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w') as file:
    file.write(data)

with open('hamlet.txt', 'r') as file:
    text = file.read().lower()


tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
total_words



ImportError: cannot import name 'layers' from 'tensorflow.keras.models' (c:\Users\tariq\Desktop\NLP MLO\venv\Lib\site-packages\keras\_tf_keras\keras\models\__init__.py)

In [ ]:
input_sequence = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[: i +1]
        input_sequence.append(n_gram_sequence)

In [ ]:
max_length = max([len(x) for x in input_sequence])
max_length

14

In [ ]:
input_sequence = np.array(pad_sequences(input_sequence, max_length, padding = 'pre'))

In [ ]:
x, y = input_sequence[:, : -1], input_sequence[: , -1]
y = tf.keras.utils.to_categorical(y, num_classes = total_words)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate = 1e-3)
loss = tf.keras.optimizers.CategoricalCrossEntropy()
earlystopping = EarlyStopping(monitor = 'val_loss', patience = 5, restore_best_weights = True)

class BahdanauAttention(layers.Layer):
    def __init__(self, attn_units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W1 = layers.Dense(attn_units)
        self.W2 = layers.Dense(attn_units)
        self.V  = layers.Dense(1)

    def call(self, values):

        score = self.V(tf.nn.tanh(self.W1(values)))
        weights = tf.nn.softmax(score, axis = 1)
        context = tf.reduce_sum(weights * values, axis = 1)
        return context 
    

def create_model(
        total_words:int, 
        max_len:int,
        embed_dim:int = 100,
        lstm_units:int = 128,
        lstm_layers:int = 1, 
        dropout:float = 0.2,
        recurrent_dropout:float = 0.0,
        bidirectional:bool = True,
        attn_units:int = 64,
        learning_rate:float = 1e-3
):
    
        inputs = layers.Input(shape =(max_len,), ndtype = 'int32')

        x = layers.Embedding(input_dim = total_words, output_dim = embed_dim, input_length = max_len)(inputs)

        for i in range(lstm_layers):
             return_seq = True
             lstm = layers.LSTM(
                  lstm_units, 
                  return_sequences = return_seq,
                  dropout = dropout if dropout > 0 else 0.0,
                  recurrent_dropout = recurrent_dropout if recurrent_dropout > 0 else 0.0,
             )
             x = layers.Bidirectional(lstm)(x) if bidirectional else lstm(x)

        context = BahdanauAttention(attn_units = attn_units)(x)

        outputs = layers.Dense(total_words, activation = 'softmax')(context)

        model = Model(inputs, outputs)

        model.compile(optimizer = opt, loss = loss, metrics = ["accuracy"])
        return model 


model = KerasClassifier(model = create_model, total_words = total_words, max_len = max_length, verbose = 0, callbacks = [earlystopping])

param_dist = {
    "model__embed_dim": [100, 128, 256],
    "model__lstm_units": randint(64, 256),
    "model__lstm_layers": [1, 2],
    "model__attn_units": [32, 64, 128],
    "model__dropout": [0.2],
    "model__learning_rate": loguniform(1e-4, 1e-2),
    "batch_size": [32, 64, 128],
    "epochs": [20]
    
}

search = RandomizedSearchCV(estimator = model, param_distributions = param_dist, n_iter = 15, cv = 3, n_jobs = 4)
search_result = search.fit(x_train, y_train)

print("Best %f using %s " (search_result.best_score_, search_result.best_params_))
